# Modelo EGARCH: Entrenamiento y Evaluación

In [21]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.egarch_wrapper import EGARCHWrapper
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer

from src.utils import trainer_toolkit as tt
from src.utils import test_toolkit as tk

warnings.filterwarnings("ignore")

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "egarch"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

print(f"Train: {len(y_train)} obs ({df_train.index[0].date()} → {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs ({df_test.index[0].date()} → {df_test.index[-1].date()})")

Train: 2171 obs (2014-01-03 → 2022-11-03)
Test:  953 obs (2022-11-07 → 2026-08-21)


## Introducción

El modelo ARCH de Engle (1982) y su generalización GARCH de Bollerslev (1986) modelan la varianza condicional como una función lineal de los choques pasados al cuadrado. Esta formulación presenta dos limitaciones: (i) requiere restricciones de no negatividad sobre los parámetros para garantizar varianzas positivas y (ii) responde de forma simétrica a choques positivos y negativos de igual magnitud, dado que el signo se pierde al elevar al cuadrado.

Nelson (1991) propuso el modelo GARCH Exponencial (EGARCH) para superar ambas limitaciones. El modelo especifica la dinámica sobre el **logaritmo** de la varianza condicional, lo que garantiza $\sigma_t^2 > 0$ sin restricciones sobre los parámetros, e incorpora un término que depende del **signo** del choque, lo que permite capturar respuestas asimétricas de la volatilidad.

Esta libreta presenta, entrena y evalúa el modelo EGARCH bajo el mismo marco experimental empleado para el modelo ARCH, con el fin de que los resultados sean directamente comparables.

### Fundamentos

Al igual que en el modelo ARCH, los retornos logarítmicos diarios se representan como $r_t = \varepsilon_t = \sigma_t z_t$, con media condicional cero y $z_t$ una innovación i.i.d. estandarizada (media cero y varianza unitaria) que sigue una distribución Normal, $t$ de Student o GED.

### Modelo EGARCH

La varianza condicional se modela según el modelo EGARCH(p, q):

$$ \ln \sigma_t^2 = \omega + \sum_{i=1}^{q} \Big[ \alpha_i \big( |z_{t-i}| - \mathbb{E}|z| \big) + \gamma_i \, z_{t-i} \Big] + \sum_{j=1}^{p} \beta_j \ln \sigma_{t-j}^2, $$

donde $z_{t-i} = \varepsilon_{t-i} / \sigma_{t-i}$ es la innovación estandarizada. Cada término cumple una función específica:

- **Efecto magnitud** ($\alpha_i$): la desviación de $|z_{t-i}|$ respecto a su valor esperado incrementa la log-varianza cuando el choque es mayor de lo habitual, de forma análoga a los coeficientes $\alpha_i$ del ARCH.
- **Efecto signo** ($\gamma_i$): introduce la asimetría. Un choque negativo ($z<0$) tiene un impacto $\alpha_i - \gamma_i$ y uno positivo un impacto $\alpha_i + \gamma_i$. Un valor $\gamma_i < 0$ representa el **efecto apalancamiento** (*leverage effect*): las caídas de precio elevan la volatilidad más que las subidas de igual magnitud.
- **Persistencia** ($\beta_j$): determina la memoria de la log-varianza. La condición de estacionariedad es $\left|\sum_j \beta_j\right| < 1$ y, en el caso EGARCH(1,1), la vida media de un choque es $\ln(0.5)/\ln(\beta_1)$ días.

Dado que la ecuación se define sobre $\ln \sigma_t^2$, los parámetros $\omega$, $\alpha_i$ y $\gamma_i$ no requieren restricciones de signo. A diferencia del ARCH, $\omega$ no representa un nivel basal de varianza, sino el intercepto de la log-varianza.


En esta libreta, $q$ denota el número de rezagos de los términos de innovación (magnitud $\alpha_i$ y signo $\gamma_i$) y $p$ el número de rezagos de la log-varianza ($\beta_j$), siguiendo la notación GARCH(p,q) de Bollerslev (1986). La librería `arch` utiliza la convención inversa: su argumento `p` corresponde a los términos simétricos, `o` a los asimétricos y `q` a los rezagos de la varianza. El *wrapper* `EGARCHWrapper` realiza esta traducción internamente (`p=q`, `o=q`, `q=p`).

La librería `arch` centra el término de magnitud con la constante $\mathbb{E}|z| = \sqrt{2/\pi}$ para todas las distribuciones. El pronóstico fuera de muestra del *wrapper* utiliza la misma constante para reproducir exactamente la recursión del modelo estimado; la diferencia respecto al valor teórico de la distribución $t$ o GED queda absorbida por el intercepto $\omega$.


La implementación computacional del modelo EGARCH se realiza con la librería `arch` de Python en su versión `8.0.0`.

## Diseño Experimental

### Marco General

Para garantizar la comparabilidad entre las familias de modelos (econométricos, ML y DL), se establece una regla de paridad de información. En cualquier instante de tiempo $t$, todos los modelos tendrán acceso exclusivamente a los retornos logarítmicos históricos del crudo Brent (la información disponible en $\mathcal{F}_{t-1}$). Se excluye el uso de variables exógenas, el precio del activo y la volatilidad realizada durante la fase de ajuste, previniendo el sesgo de anticipación (*look-ahead bias*).

La variable objetivo para los 10 modelos es la volatilidad realizada. En la fase de evaluación, las predicciones de cada modelo se contrastan contra el estimador de volatilidad realizada de Rogers-Satchell (`rs_daily`), el cual actúa como proxy observable de la varianza. Dicha serie se utiliza para el cálculo de las métricas de error y funciones de pérdida aplicadas a todos los modelos.

La función de pérdida QLIKE (por sus siglas en inglés *Quasi-Likelihood*) se emplea para evaluar pronósticos de volatilidad. Dados $\sigma_t^2$ como el proxy de volatilidad realizada y $\hat{\sigma}_t^2$ como la varianza condicional pronosticada, la función se define como:

$$
\text{QLIKE}(\sigma_t^2, \hat{\sigma}_t^2) = \frac{1}{T} \sum_{t=1}^{T} \left( \frac{\sigma_t^2}{\hat{\sigma}_t^2} - \ln\left(\frac{\sigma_t^2}{\hat{\sigma}_t^2}\right) - 1 \right)
$$

La función QLIKE presenta una estructura asimétrica, otorgando una mayor penalización a la subestimación de la volatilidad frente a su sobreestimación. De acuerdo con Patton (2011), el uso de esta función garantiza que el ordenamiento de los modelos no se altere por errores de medición en la proxy de volatilidad.

La selección de la configuración óptima se realiza identificando la combinación de hiperparámetros que **minimiza** la pérdida QLIKE promedio. Es fundamental precisar que el cálculo de esta métrica se efectúa exclusivamente sobre las predicciones *out-of-fold*, es decir, sobre el bloque de validación de cada pliegue. Esto garantiza que la selección del modelo se base en su capacidad predictiva sobre datos no vistos durante la fase de estimación de parámetros, evitando el sobreajuste y la fuga de información.

### Entrenamiento y Validación Cruzada

El ajuste de los modelos y la búsqueda de hiperparámetros se realizarán mediante un esquema de validación cruzada de ventana expansiva (*walk-forward validation*). A diferencia de la validación cruzada tradicional, este método respeta la dependencia temporal de los datos: el conjunto de entrenamiento se expande progresivamente, mientras que el conjunto de validación permanece como el bloque temporal inmediatamente posterior.

Para garantizar un tamaño muestral mínimo y evitar estimaciones inestables, se establece un periodo de calentamiento (*warm-up*) inicial. Posteriormente, los datos se dividen en 5 pliegues cronológicos. En cada iteración, el modelo se estima sobre el bloque de entrenamiento y se evalúa con la pérdida QLIKE sobre el bloque de validación.

Dado que en los modelos econométricos la estimación de parámetros se realiza mediante Máxima Verosimilitud (MLE), un proceso determinista que no depende de inicializaciones aleatorias, la obtención de ejecuciones independientes requiere la alteración del punto de origen del *walk-forward*. Por lo tanto, se ejecutará el proceso 5 veces desplazando el inicio del *warm-up* para generar particiones temporales distintas. Los resultados de cada ejecución se almacenarán de forma individual, permitiendo el cálculo de la media y la desviación estándar de las métricas para cada configuración.

La selección de la configuración óptima se realiza identificando la combinación de hiperparámetros que **minimiza** la pérdida QLIKE promedio. Es fundamental precisar que el cálculo de esta métrica se efectúa exclusivamente sobre las predicciones *out-of-fold*, es decir, sobre el bloque de validación de cada pliegue. Esto asegura que la selección del modelo se base en su capacidad predictiva sobre datos no vistos durante la fase de estimación de parámetros, previniendo el sobreajuste y la fuga de información.

Una vez concluida la fase de validación y seleccionada la combinación de hiperparámetros que minimiza la pérdida QLIKE *out-of-fold*, se procede a una reestimación final del modelo. Dicha reestimación se realiza utilizando la totalidad del conjunto de entrenamiento disponible hasta el punto de corte temporal definido. 

## Entrenamiento

Para identificar los mejores hiperparámetros del modelo, se ejecutaron cinco iteraciones independientes del esquema de validación cruzada definido para el modelo ARCH. En cada iteración se realizó una búsqueda en cuadrícula sobre el orden de innovación $q \in \{1, 2, 3\}$, el orden de persistencia $p \in \{1, 2, 3\}$ y la distribución de las innovaciones (Normal, $t$ de Student y GED), para un total de 27 configuraciones. El rango de órdenes es menor que en el ARCH porque el término autorregresivo $\beta_j \ln \sigma^2_{t-j}$ resume la memoria de la volatilidad y hace innecesarios rezagos largos de las innovaciones. El optimizador convergió en las 27 configuraciones de las cinco réplicas y ninguna combinación fue descartada por no convergencia o no estacionariedad.

In [22]:
scorer = VolatilityScorer()

for exec_id in range(1, 6):
    current_warmup = 1200 + (exec_id * 10)
    exec_dir = RESULTS_PATH / f"exec_{exec_id}"
    exec_dir.mkdir(parents=True, exist_ok=True)

    trainer = WalkForwardTrainer(
        model_class=EGARCHWrapper,
        param_grid={
            "p": [1, 2, 3],
            "q": [1, 2, 3],
            "dist": ["normal", "t", "ged"]
        },
        scorer=scorer,
        output_dir=exec_dir,
        n_splits=5,
        warmup=current_warmup
    )

    trainer.run(y_train, rs_train)

Grid Search:   0%|          | 0/27 [00:00<?, ?config/s]

Grid Search:   0%|          | 0/27 [00:00<?, ?config/s]

Grid Search:   0%|          | 0/27 [00:00<?, ?config/s]

Grid Search:   0%|          | 0/27 [00:00<?, ?config/s]

Grid Search:   0%|          | 0/27 [00:00<?, ?config/s]

Las cinco mejores configuraciones según la media utilizaron la distribución $t$ de Student. La configuración EGARCH(3,1)-$t$ obtuvo la menor pérdida (0.516 ± 0.003), seguida de un grupo de cuatro configuraciones prácticamente empatadas en 0.520, entre ellas el modelo más parsimonioso EGARCH(1,1)-$t$. La diferencia entre la primera y la quinta posición (0.004) fue del mismo orden que la desviación estándar entre réplicas, lo que indica que estas configuraciones no se distinguen en términos prácticos. La prevalencia de la distribución $t$ es congruente con las colas pesadas documentadas en el EDA y con el resultado obtenido para el modelo ARCH.

La desviación estándar dentro de cada réplica (≈ 0.19–0.20) fue alta respecto a la variación entre réplicas. Esta dispersión proviene de la heterogeneidad entre pliegues: el cuarto pliegue, que cubre el periodo del 30 de julio de 2021 al 17 de marzo de 2022 e incluye el inicio del conflicto en Ucrania, registró pérdidas entre 0.77 y 0.82 en las configuraciones con distribución $t$ y $q = 1$, mientras que el quinto pliegue registró valores cercanos a 0.30.

In [23]:
tt.display_top_results(
    RESULTS_PATH,
    top_n=5,
    aggregate_mode="mean",
    column_names=(
        "Distribución", "Orden (p)", "Orden (q)",
        "Réplica 1", "Réplica 2", "Réplica 3", "Réplica 4", "Réplica 5",
        "Global (Media ± SD)"
    )
)

Distribución,Orden (p),Orden (q),Réplica 1,Réplica 2,Réplica 3,Réplica 4,Réplica 5,Global (Media ± SD)
t,3,1,0.519 ± 0.197,0.513 ± 0.195,0.513 ± 0.191,0.517 ± 0.192,0.518 ± 0.191,0.516 ± 0.003
t,2,1,0.522 ± 0.202,0.516 ± 0.200,0.517 ± 0.198,0.521 ± 0.198,0.522 ± 0.198,0.520 ± 0.003
t,1,3,0.522 ± 0.195,0.516 ± 0.192,0.517 ± 0.189,0.521 ± 0.190,0.522 ± 0.190,0.520 ± 0.002
t,2,3,0.523 ± 0.196,0.517 ± 0.192,0.517 ± 0.188,0.521 ± 0.189,0.522 ± 0.190,0.520 ± 0.003
t,1,1,0.523 ± 0.204,0.517 ± 0.201,0.518 ± 0.199,0.521 ± 0.199,0.522 ± 0.199,0.520 ± 0.003


La agregación mediante la mediana y el rango intercuartílico (IQR) confirmó la preferencia por la distribución $t$ y por un único rezago de innovación ($q = 1$). Las configuraciones EGARCH(1,1)-$t$, EGARCH(2,1)-$t$ y EGARCH(3,1)-$t$ empataron en 0.540 ± 0.019. El empate se explica porque los rezagos adicionales de persistencia aportaron poca información: al estimar el EGARCH(2,1)-$t$ sobre el conjunto de entrenamiento, el coeficiente $\beta_2$ resultó igual a cero y los demás parámetros coincidieron con los del EGARCH(1,1)-$t$; en el EGARCH(3,1)-$t$, las pérdidas por pliegue difirieron en menos de 0.003 salvo en el cuarto pliegue, que no determina la mediana. A diferencia del modelo ARCH, el criterio de la mediana no alteró de forma sustancial el ordenamiento obtenido con la media.

In [24]:
tt.display_top_results(
    RESULTS_PATH,
    top_n=5,
    aggregate_mode="median",
    column_names=(
        "Distribución", "Orden (p)", "Orden (q)",
        "Réplica 1", "Réplica 2", "Réplica 3", "Réplica 4", "Réplica 5",
        "Global (Mediana ± IQR)"
    )
)

Distribución,Orden (p),Orden (q),Réplica 1,Réplica 2,Réplica 3,Réplica 4,Réplica 5,Global (Mediana ± IQR)
t,2,1,0.572 ± 0.263,0.558 ± 0.235,0.539 ± 0.196,0.540 ± 0.203,0.515 ± 0.201,0.540 ± 0.019
t,1,1,0.572 ± 0.263,0.558 ± 0.235,0.539 ± 0.196,0.540 ± 0.203,0.515 ± 0.201,0.540 ± 0.019
t,3,1,0.572 ± 0.263,0.558 ± 0.235,0.539 ± 0.196,0.540 ± 0.203,0.515 ± 0.201,0.540 ± 0.019
t,1,3,0.591 ± 0.256,0.562 ± 0.244,0.561 ± 0.197,0.561 ± 0.204,0.535 ± 0.209,0.561 ± 0.001
t,2,3,0.591 ± 0.261,0.561 ± 0.243,0.561 ± 0.195,0.561 ± 0.202,0.535 ± 0.208,0.561 ± 0.001


La evolución de la pérdida QLIKE en función de los órdenes mostró un patrón distinto al del modelo ARCH. En el ARCH, la pérdida disminuyó al aumentar $q$ hasta un codo entre $q = 5$ y $q = 6$. En el EGARCH, el mínimo se alcanzó con $q = 1$ para las tres distribuciones y la pérdida aumentó al incorporar más rezagos de innovación. El orden de persistencia $p$ tuvo un efecto menor, con variaciones inferiores a 0.01 dentro de cada distribución. La separación entre distribuciones fue el factor dominante: la distribución $t$ registró las pérdidas más bajas en todos los órdenes, seguida de la GED y la Normal. Estos resultados indican que un único rezago de innovación, combinado con la persistencia de la log-varianza, captura la dinámica relevante de la volatilidad.

In [25]:
tt.plot_metric_evolution(
    results_path=RESULTS_PATH,
    x_col="q",
    metric="qlike",
    aggregate_mode="mean",
    title="Evolución del QLIKE por Orden (q) y Distribución",
    xlab="Orden de innovación (q)",
    ylab="Pérdida QLIKE (Media ± SE)",
    legend_title="Distribución",
    pads=(10, 10, 15),
    label_size=11,
    title_size=14
)

tt.plot_metric_evolution(
    results_path=RESULTS_PATH,
    x_col="p",
    metric="qlike",
    aggregate_mode="mean",
    title="Evolución del QLIKE por Orden (p) y Distribución",
    xlab="Orden de persistencia (p)",
    ylab="Pérdida QLIKE (Media ± SE)",
    legend_title="Distribución",
    pads=(10, 10, 15),
    label_size=11,
    title_size=14
)

Al restringir la comparación a las configuraciones con $q = 1$, la distribución $t$ (0.540) superó a la GED (0.574–0.586) y a la Normal (0.593–0.602) en todos los órdenes de persistencia. Dado que los tres órdenes de $p$ empataron bajo la distribución $t$ y que la mejora del EGARCH(3,1)-$t$ según la media fue inferior a la variabilidad entre réplicas, se seleccionó el modelo **EGARCH(1,1)-$t$** por parsimonia. Esta especificación estima cinco parámetros ($\omega$, $\alpha_1$, $\gamma_1$, $\beta_1$, $\nu$) frente a los siete del modelo ARCH(5)-$t$.

In [26]:
tt.display_top_results(
    RESULTS_PATH,
    q_filter=[1],
    top_n=9,
    aggregate_mode="median",
    column_names=(
        "Distribución", "Orden (p)", "Orden (q)",
        "Réplica 1", "Réplica 2", "Réplica 3", "Réplica 4", "Réplica 5",
        "Global (Mediana ± IQR)"
    )
)

Distribución,Orden (p),Orden (q),Réplica 1,Réplica 2,Réplica 3,Réplica 4,Réplica 5,Global (Mediana ± IQR)
t,2,1,0.572 ± 0.263,0.558 ± 0.235,0.539 ± 0.196,0.540 ± 0.203,0.515 ± 0.201,0.540 ± 0.019
t,1,1,0.572 ± 0.263,0.558 ± 0.235,0.539 ± 0.196,0.540 ± 0.203,0.515 ± 0.201,0.540 ± 0.019
t,3,1,0.572 ± 0.263,0.558 ± 0.235,0.539 ± 0.196,0.540 ± 0.203,0.515 ± 0.201,0.540 ± 0.019
ged,3,1,0.612 ± 0.293,0.572 ± 0.289,0.574 ± 0.217,0.576 ± 0.223,0.544 ± 0.217,0.574 ± 0.004
ged,2,1,0.624 ± 0.293,0.572 ± 0.302,0.586 ± 0.216,0.589 ± 0.222,0.556 ± 0.216,0.586 ± 0.018
ged,1,1,0.624 ± 0.294,0.572 ± 0.302,0.586 ± 0.219,0.591 ± 0.222,0.561 ± 0.216,0.586 ± 0.020
normal,3,1,0.633 ± 0.335,0.580 ± 0.344,0.593 ± 0.254,0.602 ± 0.253,0.584 ± 0.222,0.593 ± 0.019
normal,2,1,0.633 ± 0.342,0.580 ± 0.352,0.593 ± 0.259,0.602 ± 0.258,0.587 ± 0.220,0.593 ± 0.015
normal,1,1,0.633 ± 0.375,0.580 ± 0.385,0.593 ± 0.290,0.602 ± 0.290,0.613 ± 0.228,0.602 ± 0.020


La tabla de estimación del modelo EGARCH(1,1)-$t$, reestimado sobre la totalidad del conjunto de entrenamiento, mostró que todos los parámetros fueron significativos según los errores estándar robustos de la librería `arch` (*p* < .001).

- *Persistencia* ($\beta_1 = 0.9845$): la log-varianza presentó una memoria larga, con una vida media de un choque de aproximadamente 44.5 días hábiles ($\ln 0.5 / \ln 0.9845$). Este valor cumple la condición de estacionariedad $|\beta_1| < 1$ y es consistente con la persistencia en varianza identificada en el EDA.
- *Efecto magnitud* ($\alpha_1 = 0.1619$): los choques de magnitud superior a la habitual incrementaron la volatilidad del día siguiente.
- *Efecto signo* ($\gamma_1 = -0.0652$): el signo negativo indicó la presencia de efecto apalancamiento. Un choque negativo tuvo un impacto de $\alpha_1 - \gamma_1 = 0.227$ sobre la log-varianza, mientras que uno positivo de igual magnitud tuvo un impacto de $\alpha_1 + \gamma_1 = 0.097$. El impacto de las caídas de precio sobre la log-varianza fue 2.3 veces mayor que el de las subidas.
- *Grados de libertad* ($\nu = 5.18$): el valor bajo confirmó las colas pesadas de las innovaciones, en línea con lo observado en el modelo ARCH ($\nu = 5.34$).


Los intervalos de $\gamma_1$ $[-0.017, 0.050]$ y de $\nu$ $[3.47, 5.03]$ no contienen la estimación puntual. El procedimiento de *block bootstrap* implementado en los *wrappers* remuestrea los residuos estandarizados por bloques y los multiplica por la volatilidad condicional original ($y^*_t = z^*_t \hat{\sigma}_t$). Esta construcción conserva la trayectoria de $\hat{\sigma}_t$, pero rompe la relación entre el signo de cada choque y la volatilidad posterior, de modo que las réplicas no contienen efecto apalancamiento y la estimación de $\gamma_1$ tiende a cero. Por esta razón, la inferencia sobre $\gamma_1$ se basó en el error estándar robusto (*p* < .001). Un bootstrap que regenere la serie de forma recursiva con los parámetros estimados corregiría esta limitación.


In [27]:
np.random.seed(42)

best_model = EGARCHWrapper(params={"p": 1, "q": 1, "dist": "t"})
best_model.fit(y_train)

df_params_boot = best_model.get_parameter_table(B=2000, decimals=4, block_size=22)

df_params_boot = df_params_boot.rename(columns={
    "Parameter": "Parámetro",
    "Coef": "Coeficiente",
    "Std. Error": "Error Estándar",
    "P-value": "p-valor",
    "CI Lower": "IC Inferior",
    "CI Upper": "IC Superior"
})

tt.display_styled_df(
    df=df_params_boot,
    decimals=4,
    p_value_cols=["p-valor"]
)

Parámetro,Coeficiente,Error Estándar,p-valor,IC Inferior,IC Superior
omega,-0.1094,0.0305,< .001,-0.1881,-0.0850
alpha[1],0.1619,0.0250,< .001,0.1355,0.2158
gamma[1],-0.0652,0.0138,< .001,-0.0173,0.0504
beta[1],0.9845,0.0040,< .001,0.9725,0.9868
nu,5.1765,0.5773,< .001,3.4717,5.0319


Las pruebas de Ljung-Box y ARCH-LM sobre los residuos estandarizados no detectaron autocorrelación ni efectos de heterocedasticidad condicional remanentes en ninguno de los rezagos evaluados, incluso antes del ajuste de Bonferroni (*p* ≥ .220 y *p* ≥ .773, respectivamente). El modelo ARCH(5)-$t$ presentaba un *p*-valor sin ajustar de .030 en la prueba ARCH-LM del rezago 30; el EGARCH(1,1)-$t$ eliminó esa dependencia residual con menos parámetros. La prueba BDS (*p* = .499) no detectó estructura no lineal remanente. Las pruebas de Jarque-Bera y Lilliefors rechazaron la normalidad de los residuos, un resultado esperado y coherente con la elección de la distribución $t$.

In [28]:
std_residuals = best_model.model_fit.std_resid

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    column_titles=(
        "Prueba Estadística",
        "Estadístico", "p-valor",
        "p-ajustado (Bonferroni)",
        "Conclusión (α = 0.05)"
    )
)

tt.display_styled_df(
    df=df_diag, decimals=3,
    p_value_cols=["p-valor", "p-ajustado"]
)

Prueba Estadística,Estadístico,p-valor,p-ajustado (Bonferroni),Conclusión (α = 0.05)
Ljung-Box (Lag 10),7.404287,.687,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),24.532633,.220,.660,Sin Autocorrelación
Ljung-Box (Lag 30),34.294049,.269,.808,Sin Autocorrelación
ARCH-LM (Lag 10),6.488147,.773,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),9.973638,.969,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),21.732098,.864,1.000,Sin Efectos ARCH
Jarque-Bera,1669.713572,< .001,< .001,Residuos No Normales
Lilliefors,0.047661,< .001,.002,Residuos No Normales
BDS (Dim 2),0.6756346389726815,.499,.499,Sin Estructura No Lineal


La figura compara la volatilidad condicional del modelo EGARCH(1,1)-$t$ con la volatilidad realizada de Rogers-Satchell en el conjunto de entrenamiento. La serie condicional reprodujo los regímenes de volatilidad identificados en el EDA: el colapso de 2014–2016, la pandemia de COVID-19 en 2020 y el conflicto en Ucrania de 2022. La alta persistencia ($\beta_1 = 0.9845$) produjo una serie suave que decayó de forma gradual después de cada episodio de tensión.

El modelo subestimó los picos extremos. Durante la pandemia de 2020, la volatilidad realizada alcanzó valores cercanos a 0.25, mientras que la volatilidad condicional llegó a un máximo cercano a 0.10. En marzo de 2022, la volatilidad realizada superó 0.11 y la condicional no superó 0.05. La formulación logarítmica responde de forma lineal a $|z_t|$, mientras que el ARCH responde de forma cuadrática a $\varepsilon_t^2$; en consecuencia, el EGARCH amortigua la reacción ante saltos aislados y privilegia la persistencia de los regímenes.

In [29]:
pred_vol_train = best_model.model_fit.conditional_volatility

tk.plot_volatility_fit(
    dates=df_train.index,
    rs=rs_train, pred_vol=pred_vol_train,
    title="Volatilidad Realizada vs Condicional (Entrenamiento)",
    legend_size=13, legend_loc="upper left"
)

## Evaluación

La tabla compara las métricas de desempeño entre el conjunto de entrenamiento (*in-sample*) y el de prueba (*out-of-sample*). La pérdida QLIKE aumentó de 0.383 en entrenamiento a 0.533 en prueba. Las métricas de error puntual (MSE, RMSE y MAE) y las métricas probabilísticas (CRPS y Pinball en los cuantiles 0.05 y 0.95) se mantuvieron en niveles similares entre ambos conjuntos.

El aumento del QLIKE se concentró en pocos días. El 9 de marzo de 2026 aportó por sí solo cerca de 0.10 al promedio del conjunto de prueba: ese día la volatilidad realizada superó 0.25 mientras el modelo pronosticaba un nivel cercano a 0.025. Al excluir los cinco días de mayor pérdida, el QLIKE de prueba descendió a 0.375, un valor inferior al de entrenamiento. Por año, la pérdida fue de 0.343 en 2023 y 0.321 en 2024, y aumentó a 0.536 en 2025 y 1.216 en 2026. De acuerdo con los umbrales de diagnóstico de sobreajuste definidos para el proyecto, la diferencia en la pérdida QLIKE ($\Delta_{abs} = 0.149$) superó el umbral de tolerancia de 0.05, lo que indica una pérdida de capacidad predictiva fuera de muestra. Las métricas de error puntual y probabilísticas, en cambio, presentaron diferencias dentro de los márgenes de estabilidad: MSE ($\Delta_{abs} \approx 10^{-6}$), RMSE ($3.7 \times 10^{-4}$), MAE ($3.6 \times 10^{-5}$), CRPS ($-5.0 \times 10^{-5}$) y Pinball en los cuantiles 0.05 y 0.95 ($-3.3 \times 10^{-5}$ y $-3.0 \times 10^{-6}$).

La pérdida de capacidad predictiva no fue generalizada, sino que se concentró en un único evento. Al excluir el 9 de marzo de 2026, la diferencia en QLIKE se redujo a 0.047, por debajo del umbral. El modelo conservó su desempeño en los periodos de fluctuación moderada, pero no generalizó ante choques alcistas extremos, una limitación asociada a la asimetría estimada que se analiza a continuación. Bajo el mismo criterio, el modelo ARCH(5)-$t$ ($\Delta_{abs} = -0.011$) no presentó pérdida de capacidad predictiva.

In [30]:
scorer = VolatilityScorer()

context = best_model.context_length
predictions_test = best_model.forecast(
    y_target=y_test,
    initial_context=y_train[-context:]
)

tk.display_train_test_comparison(
    model=best_model,
    scorer=scorer,
    rs_train=rs_train,
    rs_test=rs_test,
    preds_test=predictions_test,
    dist="t",
    nu=best_model.model_fit.params['nu'],
    decimals=6,
    column_titles=(
        "Métrica",
        "Entrenamiento",
        "Prueba",
        "Diferencia"
    ),
    show_diff=True
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.383343,0.532526,+0.149183
MSE,0.000003,0.000005,+0.000001
RMSE,0.001828,0.002199,+0.000371
MAE,0.000439,0.000476,+0.000036
CRPS,0.004720,0.004670,-0.000050
Pinball (τ=0.05),0.001853,0.001819,-0.000033
Pinball (τ=0.95),0.001791,0.001788,-0.000003


La volatilidad condicional reprodujo la tendencia general de la volatilidad realizada durante los periodos de fluctuación moderada de 2023 y 2024. La limitación estructural observada en el entrenamiento se replicó en la prueba: los saltos de junio de 2025 y marzo de 2026 no fueron anticipados, y la volatilidad condicional solo se elevó en los días posteriores: tras el salto de junio de 2025 no superó 0.034, y tras el de marzo de 2026 alcanzó un máximo de 0.065 en abril.

El día de mayor pérdida (9 de marzo de 2026) estuvo precedido por un retorno logarítmico de +8.2 % el 6 de marzo. El coeficiente $\gamma_1 < 0$ reduce el impacto de los choques positivos a menos de la mitad del de los negativos, por lo que el modelo apenas elevó la volatilidad pronosticada tras esa subida. El mismo patrón se observó en el pliegue de validación de 2022: el 7 de marzo de 2022 (volatilidad realizada ≈ 0.12) fue precedido por retornos logarítmicos de +7.3 % y +6.7 % los días 2 y 4 de marzo. El supuesto de apalancamiento, característico de los mercados de renta variable, no se cumple de forma general en el crudo Brent, donde las interrupciones de oferta generan subidas abruptas del precio acompañadas de alta volatilidad. La asimetría estimada sobre el periodo de entrenamiento, dominado por caídas como la de 2020, penalizó al modelo frente a los choques alcistas del periodo de prueba.

In [31]:
pred_vol_test = np.sqrt(predictions_test)

tk.plot_volatility_fit(
    dates=df_test.index,
    rs=rs_test,
    pred_vol=pred_vol_test,
    title="Volatilidad Realizada vs Condicional (Prueba)",
    legend_loc="upper left",
    legend_size=13
)